In [137]:
!pip install openai youtube-transcript-api tiktoken
!pip install sentence-transformers
!pip install psycopg2-binary pgvector sqlalchemy
!pip install nltk transformers


In [138]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [139]:
## STEP 2
# extracting transcript
from youtube_transcript_api import YouTubeTranscriptApi
import re

def get_video_id(url):
    return re.search(r"v=([^&]+)", url).group(1)

def fetch_transcript(url):
    video_id = get_video_id(url)
    print(video_id)
    ytt_api = YouTubeTranscriptApi()
    transcript = ytt_api.fetch(video_id)
    snippet_count = len(transcript)
    print(snippet_count)
    # full_text = " ".join([snippet.text for snippet in transcript])

    return video_id, transcript


In [140]:
# url = "https://www.youtube.com/watch?v=7xTGNNLPyMI"
# ytt_api = YouTubeTranscriptApi()
# fetched_transcript = ytt_api.fetch("7xTGNNLPyMI")
# https://www.youtube.com/watch?v=AOQyRiwydyo  -------> langchain
id, raw_transcript = fetch_transcript("https://www.youtube.com/watch?v=AOQyRiwydyo")


AOQyRiwydyo
7305


In [141]:
print(raw_transcript[0])

FetchedTranscriptSnippet(text='I am in the data domain. Why do I need', start=0.08, duration=4.32)


In [142]:
def convert_transcript(snippets):

    transcript = [
        {
            "text": snippet.text,
            "start": snippet.start
        }
        for snippet in snippets
    ]

    return transcript

transcript = convert_transcript(raw_transcript)

In [143]:
print(transcript)

[{'text': 'I am in the data domain. Why do I need', 'start': 0.08}, {'text': 'to learn lang chain in? So in 206, every', 'start': 1.6}, {'text': 'other organization is looking for the', 'start': 4.4}, {'text': 'data engineers who know lang chain', 'start': 6.08}, {'text': 'because they want you to build their AI', 'start': 7.919}, {'text': 'orchestration pipelines as well. So I', 'start': 10.32}, {'text': 'know now you will say hey I also want to', 'start': 12.32}, {'text': 'learn lang chain but I do not have any', 'start': 14.4}, {'text': 'kind of resources any structure any', 'start': 15.839}, {'text': "guide nothing don't worry that's why I", 'start': 18.08}, {'text': 'have created this complete 5 hours long', 'start': 20.16}, {'text': 'beginnerfriendly lang chain full course', 'start': 22.72}, {'text': 'which will literally hold your hand and', 'start': 25.039}, {'text': 'will make you a pro lchain developer by', 'start': 27.119}, {'text': 'the end of this video. So first you will'

In [144]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def build_chunks(sentences, max_tokens=512):

    chunks = []

    current_chunk_text = []
    current_tokens = 0
    start_time = None
    end_time = None

    for sent in sentences:

        sentence = sent["text"]
        timestamp = sent["start"]

        token_count = len(
            tokenizer.encode(sentence, add_special_tokens=False)
        )

        # initialize start time
        if start_time is None:
            start_time = timestamp

        # ✅ add sentence if still under limit
        if current_tokens + token_count <= max_tokens:
            current_chunk_text.append(sentence)
            current_tokens += token_count
            end_time = timestamp

        else:
            # save full chunk
            chunks.append({
                "text": " ".join(current_chunk_text),
                "start_time": start_time,
                "token_count": current_tokens
            })

            # start new chunk
            current_chunk_text = [sentence]
            current_tokens = token_count
            start_time = timestamp
            end_time = timestamp

    # add last chunk
    if current_chunk_text:
        chunks.append({
            "text": " ".join(current_chunk_text),
            "start_time": start_time,
            "token_count": current_tokens
        })

    return chunks


In [145]:
chunks = build_chunks(transcript)

print(len(chunks))
print(chunks[0]["token_count"])
print(chunks[0]["text"][:300])


116
509
I am in the data domain. Why do I need to learn lang chain in? So in 206, every other organization is looking for the data engineers who know lang chain because they want you to build their AI orchestration pipelines as well. So I know now you will say hey I also want to learn lang chain but I do no


In [146]:
print(chunks[1]["token_count"])
print(chunks[1]["text"][:300])

511
is in demand. So let's say you are applying for a position let's say data engineering position or any position like in the data domain. Okay. Now there are obviously hundreds or even thousands of applicants applying for the same position. But if you add these kinds of frameworks especially lang chai


In [147]:
print(chunks[-1]["token_count"])
print(chunks[-1]["text"][:300])

402
distance. But you don't need to worry about that. Just imagine that LLM is doing that all of that stuff for you like all the similarity search right. Okay. And in lang as well we have like dedicated function called similarity search. So we do not need to actually create that function because every u


In [148]:
print(chunks[-2])

{'text': "right? So it has these three things okay under the hood. So now what will happen? This LLM will go here. LLM will actually go to this thing. So when it is going to my data actually it is going to this particular thing. Make sense? So I can change the angle now. So this is the reality of it. Okay. So it is actually talking to this. So what it does it will convert your question as well in the form of vector. Okay. Hm interesting and it is called query vector because this is the query right now this query will go now this query will go inside this particular area and it will find the relevant relevant vector. So this goes inside this let's say this and it found that this query is similar or let's say it stays out but it can just go and check that this particular thing is similar. How does it know this? Because LLM knows it through the formula that it uses and what kind of formula it uses it uses something called as cosine similarity. It can be Ukraidian distance as well but do n

In [149]:
## Embedding generation
!pip install openai


In [150]:
from openai import OpenAI
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])



In [151]:
def get_embedding(text):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding


In [152]:
def embed_chunks(chunks):

    embedded_chunks = []

    for chunk in chunks:

        embedding = get_embedding(chunk["text"])

        embedded_chunks.append({
            "text": chunk["text"],
            "start_time": chunk["start_time"],
            "token_count": chunk["token_count"],
            "embedding": embedding
        })

    return embedded_chunks


In [153]:
embedded_chunks = embed_chunks(chunks)

print(len(embedded_chunks[0]["embedding"]))


1536


In [102]:
embedded_chunks[0]

{'text': "hi everyone so I've wanted to make this video for a while it is a comprehensive but General audience introduction to large language models like Chachi PT and what I'm hoping to achieve in this video is to give you kind of mental models for thinking through what it is that this tool is it is obviously magical and amazing in some respects it's uh really good at some things not very good at other things and there's also a lot of sharp edges to be aware of so what is behind this text box you can put anything in there and press enter but uh what should we be putting there and what are these words generated back how does this work and what what are you talking to exactly so I'm hoping to get at all those topics in this video we're going to go through the entire pipeline of how this stuff is built but I'm going to keep everything uh sort of accessible to a general audience so let's take a look at first how you build something like chpt and along the way I'm going to talk about um yo

In [154]:
!pip install faiss-cpu


In [155]:
import faiss
import numpy as np

embeddings_matrix = np.array(
    [chunk["embedding"] for chunk in embedded_chunks]
).astype("float32")

embeddings_matrix.shape


(116, 1536)

In [156]:
faiss.normalize_L2(embeddings_matrix)
dimension = embeddings_matrix.shape[1]

index = faiss.IndexFlatIP(dimension)  # inner product = cosine similarity
index.add(embeddings_matrix)


In [157]:
metadata_store = embedded_chunks


In [158]:
def embed_query(query):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    return np.array(response.data[0].embedding).astype("float32")


In [159]:
SIMILARITY_THRESHOLD = 0.30
def search(query, top_k=5):

    query_vector = embed_query(query)
    query_vector = np.expand_dims(query_vector, axis=0)

    faiss.normalize_L2(query_vector)

    scores, indices = index.search(query_vector, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if score < SIMILARITY_THRESHOLD:
            continue

        chunk = metadata_store[idx]

        results.append({
            "score": float(score),
            "text": chunk["text"],
            "start_time": chunk["start_time"]
        })

    return results


In [160]:
def ask(question):

    docs = search(question)

    if not docs:
        return "Question is irrelevant"

    context = "\n".join([d["text"] for d in docs])

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "Answer only from context."},
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion:{question}"
            }
        ]
    )

    return response.choices[0].message.content


In [161]:
ask("What is Langchain")

'Langchain is a popular and mature open-source framework that facilitates the building and integration of AI agentic solutions, especially in the data domain. It allows developers, including data engineers, to create AI orchestration pipelines by packaging Python classes, functions, and variables into reusable modules. Langchain is well integrated with various tools and technologies and supports capabilities in both Python and TypeScript. It is considered an early mover in the AI agents industry and helps automate complex workflows by chaining together functions like prompt templates, large language model (LLM) calls, and custom runnables, making it easier to build intelligent AI solutions efficiently.'